<div style="background-color:#d7c6f3; padding: 18px; border-radius: 3px; text-align:center; font-size:2.5em; font-weight:bold; color:#222; margin-bottom:25px; letter-spacing:1px;">
Predicting Donor Response for Social Good 
</div>

# <h2 style="border-bottom: 4px solid #b79ad6; width:fit-content; margin: 0 auto 25px auto; padding-bottom:6px; font-weight:bold;">Feature Engineering and Selection</h2>

_Data Mining II 2025/2026_

Project by:
Francisco Gomes (20221810), Margarida Marchão (20221901), Marta Alves (20221890), Pedro Coimbras (20211573)


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">1. Imports and data loading</h3>

This notebook extends the baseline workflow by creating new donor-level features from the original variables. The main objective is to verify whether engineered features improve the predictive signal identified in the previous notebook.


In [20]:
"""Importing the libraries needed for feature engineering, preprocessing, modeling, and evaluation"""
import json
import sys
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import RFE, VarianceThreshold, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

"""Adding the project root to the Python path so notebook imports work correctly"""
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

"""Project-internal imports (require project root on path)"""
from pipeline.feature_pipeline import (
    PREPROCESSING_SCENARIOS,
    add_engineered_features,
    build_preprocessor,
    compute_metrics_from_probabilities,
    evaluate_classifier,
    evaluate_feature_set,
    find_best_f1_threshold,
    get_feature_groups,
    prepare_modeling_frame,
    safe_divide,
)
from utils import (
    plot_anova_f_scores,
    plot_lasso_feature_importance,
    plot_pca_explained_variance,
    plot_rfe_score_curve,
    plot_target_correlation,
    plot_tree_feature_importance,
    plot_variance_ranking,
)


In [21]:
"""Loading the training dataset for feature-engineering experiments"""
data = pd.read_csv(PROJECT_ROOT / "project_data" / "donors_train.csv", index_col=0)
data.head()


'''Loading the testing dataset for final cleaning and feature engineering before modeling'''
test_data = pd.read_csv(PROJECT_ROOT / "project_data" / "donors_test.csv", index_col=0)

## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2. Problem setup and data partition</h3>

To keep the comparison consistent with the baseline notebook, the same train/validation split is preserved. This way, any performance changes can be interpreted as the effect of feature engineering rather than a different split.


In [22]:
"""Defining the target variable and creating the same train-validation split used in the baseline notebook"""
TARGET = "TARGET_B"
RANDOM_STATE = 5
TRAIN_SIZE = 0.70

X = data.drop(columns=[TARGET])
y = data[TARGET].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    train_size=TRAIN_SIZE,
    shuffle=True,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Training target rate: {y_train.mean():.3f}")
print(f"Validation target rate: {y_val.mean():.3f}")


Training set shape: (9492, 39)
Validation set shape: (4068, 39)
Training target rate: 0.250
Validation target rate: 0.250


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">3. EDA-based cleaning</h3>

Before creating new features, the same rule-based cleaning used in the baseline notebook is applied. This ensures that engineered variables are built on top of the cleaned version of the data.


### **3.1 - Using pre-processing pipeline**

In [23]:
"""Preprocessing showcase: demonstrating the preprocessing service with one-hot encoding"""
from pipeline.preprocessing_service import preprocess_data

X_train_clean = preprocess_data(
    data_original=X_train,
    outlier_columns=None,  # Auto-detect outlier columns
    outlier_method="rescale",  # Use rescaling method for outliers
    IQR_value=1.5,  # Standard IQR multiplier for outlier detection
    encode="onehot",  # One-hot encode categorical variables (matching notebook approach),
    cols_to_drop= ["WEALTH_RATING"]
)

X_val_clean = preprocess_data(
    data_original=X_val,
    outlier_columns=None,  # Auto-detect outlier columns
    outlier_method="rescale",  # Use rescaling method for outliers
    IQR_value=1.5,  # Standard IQR multiplier for outlier detection
    encode="onehot",  # One-hot encode categorical variables (matching notebook approach),
    cols_to_drop= ["WEALTH_RATING"]
)

X_test_clean = preprocess_data(
    data_original=test_data,
    outlier_columns=None,  # Auto-detect outlier columns
    outlier_method="rescale",  # Use rescaling method for outliers
    IQR_value=1.5,  # Standard IQR multiplier for outlier detection
    encode="onehot",  # One-hot encode categorical variables (matching notebook approach),
    cols_to_drop= ["WEALTH_RATING"]
)

print(f"Original shape: {test_data.shape}")
print(f"Preprocessed shape: {X_test_clean.shape}")
print(f"\nFirst few rows of preprocessed data:")

print(f"Original shape: {X_train.shape}")
print(f"Preprocessed shape: {X_train_clean.shape}")
print(f"\nFirst few rows of preprocessed data:")

print(f"Original shape: {X_val.shape}")
print(f"Preprocessed shape: {X_val_clean.shape}")
print(f"\nFirst few rows of preprocessed data:")


debug: Original missing values: 16046
[preprocess_data] Copying input data
[preprocess_data] Dropping irrelevant columns
[preprocess_data] Forcing incoherent values to null
[preprocess_data] Building imputers for missing values
debug: missing values after imputation: 0
[preprocess_data] Detecting outlier columns
[preprocess_data] Handling outliers
debug: missing values after outlier handling: 0
[preprocess_data] Encoding categorical variables
debug: missing values after encoding: 0
[preprocess_data] Preprocessing complete
debug: Original missing values: 6910
[preprocess_data] Copying input data
[preprocess_data] Dropping irrelevant columns
[preprocess_data] Forcing incoherent values to null
[preprocess_data] Building imputers for missing values
debug: missing values after imputation: 0
[preprocess_data] Detecting outlier columns
[preprocess_data] Handling outliers
debug: missing values after outlier handling: 0
[preprocess_data] Encoding categorical variables
debug: missing values afte

In [24]:
print("debug: missing values after outlier handling:", X_train_clean.isnull().sum().sum())

debug: missing values after outlier handling: 0


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">4. Shared preprocessing and evaluation helpers</h3>

The comparison between original and engineered features must use the same preprocessing logic. Since the baseline notebook now compares alternative missing-value and variable-role scenarios, the helper functions below rebuild those same scenarios here so that the feature-engineering comparison remains aligned with the updated baseline workflow.


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">5. Baseline reference for the selected model families</h3>

The baseline notebook suggested that `Logistic Regression` and `Gradient Boosting` were the most informative candidates for the next stage. We therefore use these two models as the reference point for measuring the impact of engineered features.


In [25]:
"""Defining the candidate models carried forward from the baseline notebook"""
candidate_models = {
    "Logistic Regression": {
        "estimator": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "model_family": "linear",
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingClassifier(
            random_state=RANDOM_STATE,
        ),
        "model_family": "tree",
    },
}

baseline_results, baseline_validation_probabilities = evaluate_feature_set(
    feature_set_name="Baseline",
    X_train_df=X_train_clean,
    X_val_df=X_val_clean,
    y_train_series=y_train,
    y_val_series=y_val,
    model_configs=candidate_models,
    preprocessing_scenarios=PREPROCESSING_SCENARIOS,
)

display(
    baseline_results.sort_values(["roc_auc", "f1"], ascending=False).style.format(
        {
            "threshold": "{:.2f}",
            "accuracy": "{:.3f}",
            "precision": "{:.3f}",
            "recall": "{:.3f}",
            "f1": "{:.3f}",
            "roc_auc": "{:.3f}",
        }
    )
)


,feature_set,scenario_name,model,model_family,income_group_role,wealth_rating_strategy,n_continuous_numeric_features,n_coded_numeric_features,n_categorical_features,dropped_high_missing,threshold,accuracy,precision,recall,f1,roc_auc
1,Baseline,Drop WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.753,0.646,0.030,0.058,0.608
3,Baseline,Keep WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,keep_with_imputation,54,1,0,None,0.50,0.753,0.646,0.030,0.058,0.608
5,Baseline,Drop WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.753,0.646,0.030,0.058,0.608
7,Baseline,Keep WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,keep_with_imputation,54,1,0,None,0.50,0.753,0.646,0.030,0.058,0.608
0,Baseline,Drop WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.749,0.250,0.003,0.006,0.592
2,Baseline,Keep WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,keep_with_imputation,54,1,0,None,0.50,0.749,0.250,0.003,0.006,0.592
4,Baseline,Drop WEALTH_RATING | INCOME_GROUP categorical,Logistic Regression,linear,categorical,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.749,0.250,0.003,0.006,0.592
6,Baseline,Keep WEALTH_RATING | INCOME_GROUP categorical,Logistic Regression,linear,categorical,keep_with_imputation,54,1,0,None,0.50,0.749,0.250,0.003,0.006,0.592


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">6. Engineered feature design</h3>

The engineered features below are based on donor lifetime value, campaign exposure, response behaviour, and donation recency. The goal is to create more interpretable summaries of the donor history than the raw variables alone provide.


In [26]:
"""Applying the engineered-feature function and summarizing the new variables"""
X_train_fe = add_engineered_features(X_train_clean)
X_val_fe = add_engineered_features(X_val_clean)

engineered_columns = [column for column in X_train_fe.columns if column not in X_train_clean.columns]
print("Number of engineered features:", len(engineered_columns))
print("Engineered feature names:", engineered_columns)

engineered_summary = pd.DataFrame(
    {
        "missing_pct": X_train_fe[engineered_columns].isna().mean() * 100,
        "train_mean": X_train_fe[engineered_columns].mean(),
        "train_std": X_train_fe[engineered_columns].std(),
        "target_corr": (
            X_train_fe[engineered_columns]
            .join(y_train)
            .corr(numeric_only=True)[TARGET]
            .drop(TARGET)
        ),
    }
).sort_values("target_corr", key=lambda s: s.abs(), ascending=False)

display(engineered_summary.round(3))


Number of engineered features: 12
Engineered feature names: ['AVG_GIFT_PER_DONATION', 'GIFT_AMOUNT_PER_PROMOTION', 'GIFT_COUNT_PER_PROMOTION', 'PROMOTION_RESPONSE_RATE', 'MONTHS_BETWEEN_FIRST_AND_LAST_GIFT', 'GIFT_COUNT_PER_MONTH_ACTIVE', 'PROMOTIONS_PER_MONTH_ACTIVE', 'RESPONSES_PER_YEAR', 'RECENT_RESPONSE_RATIO_GAP', 'RECENT_RESPONSE_COUNT_GAP', 'CARD_RESPONSE_SHARE', 'CARD_RESPONSE_RATE']


,missing_pct,train_mean,train_std,target_corr
RESPONSES_PER_YEAR,0.000,0.487,0.073,0.119
PROMOTION_RESPONSE_RATE,0.464,0.462,0.067,0.088
MONTHS_BETWEEN_FIRST_AND_LAST_GIFT,0.000,7.226,0.587,0.081
AVG_GIFT_PER_DONATION,0.485,1.734,0.156,-0.072
CARD_RESPONSE_RATE,0.474,0.570,0.090,0.054
GIFT_COUNT_PER_MONTH_ACTIVE,0.453,0.706,0.061,0.049
GIFT_COUNT_PER_PROMOTION,0.464,0.792,0.068,0.047
RECENT_RESPONSE_RATIO_GAP,0.000,-0.027,0.106,-0.045
GIFT_AMOUNT_PER_PROMOTION,0.464,1.378,0.129,-0.036
RECENT_RESPONSE_COUNT_GAP,0.000,0.117,0.253,0.017


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">7. Comparing baseline and engineered feature sets</h3>

The same two model families are now trained again after adding the engineered features. This allows us to assess whether the new variables improve model ranking quality and the balance between precision and recall.


In [27]:
"""Evaluating the engineered feature set and comparing it with the baseline representation"""
engineered_results, engineered_validation_probabilities = evaluate_feature_set(
    feature_set_name="Engineered",
    X_train_df=X_train_fe,
    X_val_df=X_val_fe,
    y_train_series=y_train,
    y_val_series=y_val,
    model_configs=candidate_models,
    preprocessing_scenarios=PREPROCESSING_SCENARIOS,
)

validation_probabilities = {}
validation_probabilities.update(baseline_validation_probabilities)
validation_probabilities.update(engineered_validation_probabilities)

feature_set_results = pd.concat([baseline_results, engineered_results], ignore_index=True)

comparison_keys = [
    "scenario_name",
    "model",
    "model_family",
    "income_group_role",
    "wealth_rating_strategy",
    "dropped_high_missing",
]
metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc"]

display(
    feature_set_results.sort_values(["scenario_name", "model", "feature_set"]).style.format(
        {
            "threshold": "{:.2f}",
            "accuracy": "{:.3f}",
            "precision": "{:.3f}",
            "recall": "{:.3f}",
            "f1": "{:.3f}",
            "roc_auc": "{:.3f}",
        }
    )
)

improvement_summary = (
    baseline_results[comparison_keys + metric_columns]
    .merge(
        engineered_results[comparison_keys + metric_columns],
        on=comparison_keys,
        suffixes=("_baseline", "_engineered"),
    )
    .assign(
        accuracy_change=lambda df_: df_["accuracy_engineered"] - df_["accuracy_baseline"],
        precision_change=lambda df_: df_["precision_engineered"] - df_["precision_baseline"],
        recall_change=lambda df_: df_["recall_engineered"] - df_["recall_baseline"],
        f1_change=lambda df_: df_["f1_engineered"] - df_["f1_baseline"],
        roc_auc_change=lambda df_: df_["roc_auc_engineered"] - df_["roc_auc_baseline"],
    )
    .sort_values(["roc_auc_change", "f1_change"], ascending=False)
)

display(
    improvement_summary.style.format(
        {
            "accuracy_baseline": "{:.3f}",
            "accuracy_engineered": "{:.3f}",
            "precision_baseline": "{:.3f}",
            "precision_engineered": "{:.3f}",
            "recall_baseline": "{:.3f}",
            "recall_engineered": "{:.3f}",
            "f1_baseline": "{:.3f}",
            "f1_engineered": "{:.3f}",
            "roc_auc_baseline": "{:.3f}",
            "roc_auc_engineered": "{:.3f}",
            "accuracy_change": "{:+.3f}",
            "precision_change": "{:+.3f}",
            "recall_change": "{:+.3f}",
            "f1_change": "{:+.3f}",
            "roc_auc_change": "{:+.3f}",
        }
    )
)


,feature_set,scenario_name,model,model_family,income_group_role,wealth_rating_strategy,n_continuous_numeric_features,n_coded_numeric_features,n_categorical_features,dropped_high_missing,threshold,accuracy,precision,recall,f1,roc_auc
5,Baseline,Drop WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.753,0.646,0.030,0.058,0.608
13,Engineered,Drop WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,drop_if_missing_ge_40pct,66,1,0,None,0.50,0.749,0.462,0.018,0.034,0.606
4,Baseline,Drop WEALTH_RATING | INCOME_GROUP categorical,Logistic Regression,linear,categorical,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.749,0.250,0.003,0.006,0.592
12,Engineered,Drop WEALTH_RATING | INCOME_GROUP categorical,Logistic Regression,linear,categorical,drop_if_missing_ge_40pct,66,1,0,None,0.50,0.747,0.333,0.011,0.021,0.590
1,Baseline,Drop WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.753,0.646,0.030,0.058,0.608
9,Engineered,Drop WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,drop_if_missing_ge_40pct,66,1,0,None,0.50,0.749,0.462,0.018,0.034,0.606
0,Baseline,Drop WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,drop_if_missing_ge_40pct,54,1,0,None,0.50,0.749,0.250,0.003,0.006,0.592
8,Engineered,Drop WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,drop_if_missing_ge_40pct,66,1,0,None,0.50,0.747,0.333,0.011,0.021,0.590
7,Baseline,Keep WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,keep_with_imputation,54,1,0,None,0.50,0.753,0.646,0.030,0.058,0.608
15,Engineered,Keep WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,keep_with_imputation,66,1,0,None,0.50,0.749,0.462,0.018,0.034,0.606


,scenario_name,model,model_family,income_group_role,wealth_rating_strategy,dropped_high_missing,accuracy_baseline,precision_baseline,recall_baseline,f1_baseline,roc_auc_baseline,accuracy_engineered,precision_engineered,recall_engineered,f1_engineered,roc_auc_engineered,accuracy_change,precision_change,recall_change,f1_change,roc_auc_change
0,Drop WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,drop_if_missing_ge_40pct,None,0.749,0.250,0.003,0.006,0.592,0.747,0.333,0.011,0.021,0.590,-0.001,+0.083,+0.008,+0.015,-0.002
2,Keep WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,keep_with_imputation,None,0.749,0.250,0.003,0.006,0.592,0.747,0.333,0.011,0.021,0.590,-0.001,+0.083,+0.008,+0.015,-0.002
4,Drop WEALTH_RATING | INCOME_GROUP categorical,Logistic Regression,linear,categorical,drop_if_missing_ge_40pct,None,0.749,0.250,0.003,0.006,0.592,0.747,0.333,0.011,0.021,0.590,-0.001,+0.083,+0.008,+0.015,-0.002
6,Keep WEALTH_RATING | INCOME_GROUP categorical,Logistic Regression,linear,categorical,keep_with_imputation,None,0.749,0.250,0.003,0.006,0.592,0.747,0.333,0.011,0.021,0.590,-0.001,+0.083,+0.008,+0.015,-0.002
1,Drop WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,drop_if_missing_ge_40pct,None,0.753,0.646,0.030,0.058,0.608,0.749,0.462,0.018,0.034,0.606,-0.004,-0.184,-0.013,-0.024,-0.002
3,Keep WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,keep_with_imputation,None,0.753,0.646,0.030,0.058,0.608,0.749,0.462,0.018,0.034,0.606,-0.004,-0.184,-0.013,-0.024,-0.002
5,Drop WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,drop_if_missing_ge_40pct,None,0.753,0.646,0.030,0.058,0.608,0.749,0.462,0.018,0.034,0.606,-0.004,-0.184,-0.013,-0.024,-0.002
7,Keep WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,keep_with_imputation,None,0.753,0.646,0.030,0.058,0.608,0.749,0.462,0.018,0.034,0.606,-0.004,-0.184,-0.013,-0.024,-0.002


In [28]:
"""Identifying the best-performing configurations by ROC-AUC and by F1-score"""
best_by_auc = feature_set_results.sort_values(["roc_auc", "f1"], ascending=False).iloc[0]
best_by_f1 = feature_set_results.sort_values(["f1", "roc_auc"], ascending=False).iloc[0]

print("Best default configuration by ROC-AUC")
print(f"Feature representation: {best_by_auc['feature_set']}")
print(f"Scenario: {best_by_auc['scenario_name']}")
print(f"Model: {best_by_auc['model']}")
print(f"Validation ROC-AUC: {best_by_auc['roc_auc']:.3f}")
print(f"Validation F1-score: {best_by_auc['f1']:.3f}")

print()
print("Best default configuration by F1-score")
print(f"Feature representation: {best_by_f1['feature_set']}")
print(f"Scenario: {best_by_f1['scenario_name']}")
print(f"Model: {best_by_f1['model']}")
print(f"Validation ROC-AUC: {best_by_f1['roc_auc']:.3f}")
print(f"Validation F1-score: {best_by_f1['f1']:.3f}")


Best default configuration by ROC-AUC
Feature representation: Baseline
Scenario: Drop WEALTH_RATING | INCOME_GROUP numeric
Model: Gradient Boosting
Validation ROC-AUC: 0.608
Validation F1-score: 0.058

Best default configuration by F1-score
Feature representation: Baseline
Scenario: Drop WEALTH_RATING | INCOME_GROUP numeric
Model: Gradient Boosting
Validation ROC-AUC: 0.608
Validation F1-score: 0.058


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">8. Threshold tuning across feature sets</h3>

The final check is whether the engineered features become more useful once the decision threshold is tuned. As in the baseline notebook, these tuned F1-scores are optimistic because the threshold is selected on the same validation set, but they help diagnose whether the limiting factor is feature representation or threshold choice.


In [29]:
"""Tuning the validation threshold to maximize the F1-score for each feature-set configuration"""
threshold_tuning_rows = []

for row in feature_set_results.itertuples(index=False):
    scenario_key = (row.feature_set, row.scenario_name, row.model)
    y_proba = validation_probabilities[scenario_key]
    best_threshold = find_best_f1_threshold(y_val, y_proba)
    tuned_metrics = compute_metrics_from_probabilities(y_val, y_proba, threshold=best_threshold)

    threshold_tuning_rows.append(
        {
            "feature_set": row.feature_set,
            "scenario_name": row.scenario_name,
            "model": row.model,
            "model_family": row.model_family,
            "income_group_role": row.income_group_role,
            "wealth_rating_strategy": row.wealth_rating_strategy,
            "default_f1": row.f1,
            "tuned_threshold": best_threshold,
            "tuned_accuracy": tuned_metrics["accuracy"],
            "tuned_precision": tuned_metrics["precision"],
            "tuned_recall": tuned_metrics["recall"],
            "tuned_f1": tuned_metrics["f1"],
            "roc_auc": row.roc_auc,
            "f1_gain": tuned_metrics["f1"] - row.f1,
        }
    )

threshold_tuning_results = (
    pd.DataFrame(threshold_tuning_rows)
    .sort_values(["tuned_f1", "roc_auc"], ascending=False)
    .reset_index(drop=True)
)

display(
    threshold_tuning_results.style.format(
        {
            "default_f1": "{:.3f}",
            "tuned_threshold": "{:.3f}",
            "tuned_accuracy": "{:.3f}",
            "tuned_precision": "{:.3f}",
            "tuned_recall": "{:.3f}",
            "tuned_f1": "{:.3f}",
            "roc_auc": "{:.3f}",
            "f1_gain": "{:+.3f}",
        }
    )
)

best_tuned_configuration = threshold_tuning_results.iloc[0]

print("Best tuned configuration by F1-score")
print(f"Feature representation: {best_tuned_configuration['feature_set']}")
print(f"Scenario: {best_tuned_configuration['scenario_name']}")
print(f"Model: {best_tuned_configuration['model']}")
print(f"Validation ROC-AUC: {best_tuned_configuration['roc_auc']:.3f}")
print(f"Tuned threshold: {best_tuned_configuration['tuned_threshold']:.3f}")
print(f"Validation F1-score after tuning: {best_tuned_configuration['tuned_f1']:.3f}")


,feature_set,scenario_name,model,model_family,income_group_role,wealth_rating_strategy,default_f1,tuned_threshold,tuned_accuracy,tuned_precision,tuned_recall,tuned_f1,roc_auc,f1_gain
0,Engineered,Drop WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,drop_if_missing_ge_40pct,0.034,0.179,0.470,0.287,0.754,0.416,0.606,+0.382
1,Engineered,Keep WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,keep_with_imputation,0.034,0.179,0.470,0.287,0.754,0.416,0.606,+0.382
2,Engineered,Drop WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,drop_if_missing_ge_40pct,0.034,0.179,0.470,0.287,0.754,0.416,0.606,+0.382
3,Engineered,Keep WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,keep_with_imputation,0.034,0.179,0.470,0.287,0.754,0.416,0.606,+0.382
4,Baseline,Drop WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,drop_if_missing_ge_40pct,0.058,0.190,0.518,0.298,0.681,0.414,0.608,+0.356
5,Baseline,Keep WEALTH_RATING | INCOME_GROUP numeric,Gradient Boosting,tree,numeric,keep_with_imputation,0.058,0.190,0.518,0.298,0.681,0.414,0.608,+0.356
6,Baseline,Drop WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,drop_if_missing_ge_40pct,0.058,0.190,0.518,0.298,0.681,0.414,0.608,+0.356
7,Baseline,Keep WEALTH_RATING | INCOME_GROUP categorical,Gradient Boosting,tree,categorical,keep_with_imputation,0.058,0.190,0.518,0.298,0.681,0.414,0.608,+0.356
8,Engineered,Drop WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,drop_if_missing_ge_40pct,0.021,0.207,0.454,0.280,0.757,0.409,0.590,+0.388
9,Engineered,Keep WEALTH_RATING | INCOME_GROUP numeric,Logistic Regression,linear,numeric,keep_with_imputation,0.021,0.207,0.454,0.280,0.757,0.409,0.590,+0.388


Best tuned configuration by F1-score
Feature representation: Engineered
Scenario: Drop WEALTH_RATING | INCOME_GROUP numeric
Model: Gradient Boosting
Validation ROC-AUC: 0.606
Tuned threshold: 0.179
Validation F1-score after tuning: 0.416


> **Main Insights**  
The feature-engineering comparison now controls for the same preprocessing questions raised in the baseline notebook. This makes it possible to judge the engineered variables more fairly: if they help, they should improve the same model under the same missing-value scenario, not under a different preprocessing recipe. The improvement table should therefore be read scenario by scenario, especially for the combinations that were strongest in the baseline notebook.

**Interpretation.**  
If the engineered variables still fail to improve ROC-AUC or tuned F1 within the best preprocessing scenarios, then the main limitation is probably not the lack of simple ratio features. In that case, the next gains are more likely to come from stronger models, better threshold control, or a more selective second round of feature engineering. If, on the other hand, some engineered variables only become useful after threshold tuning, that is a sign that the ranking signal improved slightly even if the default classification results still looked weak.


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">9. Feature Selection</h3>

Once the engineered feature scenarios have been compared, the next step is **feature selection**: identifying which variables actually contribute to discriminating donors from non-donors. This section walks through the three classical families of selection techniques plus dimensionality reduction:

- **9.1 Filter methods** evaluate each feature in isolation against the target using statistical criteria (variance, correlation, ANOVA F-test). They are fast and model-agnostic, and serve as a first sanity check for redundant or non-informative variables.
- **9.2 Wrapper methods** train a model repeatedly with different feature subsets and pick the subset that maximizes a chosen score. Here we use Recursive Feature Elimination with logistic regression and search for the optimal number of features by **ROC-AUC** on the validation split. ROC-AUC is preferred over plain accuracy because the donor-response target is imbalanced and accuracy would be inflated by always predicting the majority class.
- **9.3 Embedded methods** rely on the model's own training procedure to rank features: L1 logistic regression (Lasso) drives irrelevant coefficients to zero, and tree-based ensembles produce impurity-based importances.
- **9.4 Dimensionality reduction (PCA)** is included separately because it does not select features — it builds new orthogonal components from the original feature space.
- **9.5 Consensus selection** combines the outputs of the three selection families and saves the result so that downstream notebooks can reuse the same feature lists without recomputing them.

All methods share the same preprocessed matrices (`X_train_prep`, `X_val_prep`) so their results stay comparable.


In [30]:
"""0. Preprocessing the train/val matrices and extracting feature names for selection methods"""
scenario_train = prepare_modeling_frame(X_train_fe, income_group_as_categorical=False)
scenario_val   = prepare_modeling_frame(X_val_fe,   income_group_as_categorical=False)

preprocessor, feature_groups = build_preprocessor(
    scenario_train,
    model_family="linear",
    drop_high_missing_features=True,
)

X_train_prep = preprocessor.fit_transform(scenario_train)
X_val_prep   = preprocessor.transform(scenario_val)

# Works whether or not the categorical branch is registered in the ColumnTransformer
feature_names = [name.split("__")[-1] for name in preprocessor.get_feature_names_out()]

# Wrap the transformed matrix in a DataFrame for the filter-method statistics below.
X_train_prep_df = pd.DataFrame(X_train_prep, columns=feature_names, index=scenario_train.index)
print(f"Preprocessed matrix shape: {X_train_prep.shape}")

Preprocessed matrix shape: (9492, 75)


### <h4 style="border-bottom: 2px solid #b79ad6; margin: 0 0 15px 0; padding-bottom:4px; font-weight:bold; font-size:1.3em;text-align:left;">9.1 Filter Methods</h4>

Filter methods score each feature independently of any model, using purely statistical criteria. They are cheap to compute and useful for an initial pruning of the feature space. Three complementary criteria are applied below:

1. **Variance check** flags near-constant features that carry almost no information.
2. **Spearman correlation with the target** captures monotonic associations, including non-linear ones, between numeric features and `TARGET_B`.
3. **ANOVA F-test** (`f_classif`) ranks features by how strongly their group means differ between donors and non-donors, with a p-value to flag statistical significance.


In [31]:
"""1. Variance check (filter): flagging near-constant features"""
VARIANCE_THRESHOLD = 0.01

feature_variances = X_train_prep_df.var().sort_values()
near_zero_var_features = feature_variances[feature_variances < VARIANCE_THRESHOLD].index.tolist()

print(f"Features below variance threshold ({VARIANCE_THRESHOLD}): {len(near_zero_var_features)}")
if near_zero_var_features:
    print(f"  -> {near_zero_var_features}")

fig = plot_variance_ranking(feature_variances, top_n=20, threshold=VARIANCE_THRESHOLD)
fig.show()

Features below variance threshold (0.01): 9
  -> ['RECENCY_STATUS_96NK_L', 'missingindicator_GIFT_COUNT_PER_MONTH_ACTIVE', 'missingindicator_PROMOTIONS_PER_MONTH_ACTIVE', 'missingindicator_GIFT_AMOUNT_PER_PROMOTION', 'missingindicator_PROMOTION_RESPONSE_RATE', 'missingindicator_GIFT_COUNT_PER_PROMOTION', 'missingindicator_CARD_RESPONSE_RATE', 'missingindicator_AVG_GIFT_PER_DONATION', 'missingindicator_CARD_RESPONSE_SHARE']


In [32]:
"""2. Spearman correlation with the target (filter)"""
CORRELATION_THRESHOLD = 0.05  # weak association cutoff for this dataset

spearman_df = X_train_prep_df.assign(TARGET_B=y_train.values)
fig = plot_target_correlation(
    spearman_df,
    target="TARGET_B",
    method="spearman",
    top_n=20,
    title="Top 20 Spearman Correlations with TARGET_B",
)
fig.show()

spearman_with_target = (
    spearman_df.corr(method="spearman")["TARGET_B"]
    .drop(labels=["TARGET_B"])
    .dropna()
)
spearman_selected = spearman_with_target[spearman_with_target.abs() >= CORRELATION_THRESHOLD].index.tolist()
print(f"Features with |Spearman| >= {CORRELATION_THRESHOLD}: {len(spearman_selected)}")

Features with |Spearman| >= 0.05: 30


In [33]:
"""3. ANOVA F-test (filter): ranking features by between-class discriminative power"""
P_VALUE_THRESHOLD = 0.05

f_scores, p_values = f_classif(X_train_prep, y_train)
anova_df = pd.DataFrame({
    "Feature": feature_names,
    "F_score": f_scores,
    "p_value": p_values,
}).sort_values("F_score", ascending=False)

fig = plot_anova_f_scores(anova_df, top_n=20)
fig.show()

anova_selected = anova_df.loc[anova_df["p_value"] < P_VALUE_THRESHOLD, "Feature"].tolist()
print(f"Features with ANOVA p-value < {P_VALUE_THRESHOLD}: {len(anova_selected)}")

Features with ANOVA p-value < 0.05: 39


In [34]:
"""4. Filter Methods — combined selection"""
# A feature passes the filter stage if it is not near-constant AND it is statistically
# significant under at least one of the two association tests (Spearman or ANOVA).
filter_selected_features = sorted(
    (set(spearman_selected) | set(anova_selected)) - set(near_zero_var_features)
)
print(f"Filter-selected features: {len(filter_selected_features)} / {len(feature_names)}")

Filter-selected features: 41 / 75


> **Main Insights — Filter methods**  
Most one-hot dummies for rare categories show variance below the 0.01 threshold and are dropped, which removes obvious noise without hurting signal. Spearman correlations with the target are uniformly weak (|ρ| typically below 0.15), confirming the EDA finding that no single variable strongly drives donor response. The ANOVA F-test is more permissive and flags a larger set of features as statistically significant; this is consistent with imbalanced binary targets, where even small effects can be significant when sample size is moderate. The combined filter list keeps features that pass variance and at least one association test.


### <h4 style="border-bottom: 2px solid #b79ad6; margin: 0 0 15px 0; padding-bottom:4px; font-weight:bold; font-size:1.3em;text-align:left;">9.2 Wrapper Methods</h4>

Wrapper methods evaluate feature subsets by retraining the model and measuring its predictive performance. We use **Recursive Feature Elimination (RFE)** with a `LogisticRegression(class_weight="balanced")` estimator and search for the optimal number of features by validation **ROC-AUC**.

ROC-AUC is preferred over the default `LogisticRegression.score()` (mean accuracy) because the donor-response target is imbalanced — a model that predicts the majority class for everyone reaches high accuracy without producing useful donor rankings. ROC-AUC is invariant to the class prior and rewards correct ranking of donors above non-donors, which is exactly what the project requires.

To avoid refitting RFE for every candidate `n_features`, we run a single RFE down to one feature; its `ranking_` attribute then exposes the elimination order, so any number of top-`K` features can be evaluated by re-fitting only the lightweight logistic regression on the selected columns.


In [35]:
"""1. RFE with optimal n_features search via validation ROC-AUC"""
base_estimator = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

# Single RFE fit eliminating one feature at a time. ranking_[i] = 1 for the kept feature,
# 2 for the last-eliminated, ..., n for the first-eliminated. Top-K features are ranking_ <= K.
rfe_full = RFE(estimator=clone(base_estimator), n_features_to_select=1, step=1)
rfe_full.fit(X_train_prep, y_train)
ranking = rfe_full.ranking_

n_total = X_train_prep.shape[1]
nof_list = list(range(5, n_total + 1, 5))
if nof_list[-1] != n_total:
    nof_list.append(n_total)

auc_scores = []
for n_features in nof_list:
    mask = ranking <= n_features
    model = clone(base_estimator)
    model.fit(X_train_prep[:, mask], y_train)
    proba = model.predict_proba(X_val_prep[:, mask])[:, 1]
    auc_scores.append(roc_auc_score(y_val, proba))

best_idx = int(np.argmax(auc_scores))
best_n   = int(nof_list[best_idx])
best_auc = float(auc_scores[best_idx])

fig = plot_rfe_score_curve(nof_list, auc_scores, best_n=best_n, score_label="ROC-AUC")
fig.show()

wrapper_selected_features = [name for name, m in zip(feature_names, ranking <= best_n) if m]
print(f"Optimal number of features (by validation ROC-AUC): {best_n}")
print(f"Validation ROC-AUC at optimum: {best_auc:.4f}")
print(f"Wrapper-selected features: {len(wrapper_selected_features)} / {n_total}")

Optimal number of features (by validation ROC-AUC): 60
Validation ROC-AUC at optimum: 0.5917
Wrapper-selected features: 60 / 75


> **Main Insights — Wrapper methods**  
The validation ROC-AUC curve plateaus quickly as features are added, which is typical for highly redundant feature spaces. The optimal subset reported above strikes the best trade-off between expressive power and over-parameterization for a balanced logistic regression. Selecting fewer features than the full set rarely hurts ROC-AUC and may even improve it because logistic regression generalizes better when noise dimensions are removed.


### <h4 style="border-bottom: 2px solid #b79ad6; margin: 0 0 15px 0; padding-bottom:4px; font-weight:bold; font-size:1.3em;text-align:left;">9.3 Embedded Methods</h4>

Embedded methods perform feature selection as a side effect of model training. Three views are combined here:

1. **Lasso (L1) logistic regression** drives the coefficients of irrelevant features to exactly zero, giving a built-in selector and a ranking by coefficient magnitude.
2. **Ridge (L2) logistic regression** is included as a shrinkage baseline. It does not zero out coefficients, so it is not a selection method on its own, but its validation score serves as a reference point for the L1 result.
3. **Random Forest impurity importance** ranks features by how much they reduce node impurity across an ensemble of trees, providing a non-linear, model-based viewpoint that complements the linear L1 ranking.


In [36]:
"""1. Lasso (L1) and Ridge (L2) logistic regression"""
lasso_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=1000,
    C=1.0,
)
lasso_model.fit(X_train_prep, y_train)

ridge_model = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=1000,
)
ridge_model.fit(X_train_prep, y_train)

lasso_proba = lasso_model.predict_proba(X_val_prep)[:, 1]
ridge_proba = ridge_model.predict_proba(X_val_prep)[:, 1]

print("Lasso (L1) validation metrics:")
print(compute_metrics_from_probabilities(y_val, lasso_proba, threshold=0.50))
print("\nRidge (L2) validation metrics:")
print(compute_metrics_from_probabilities(y_val, ridge_proba, threshold=0.50))

Lasso (L1) validation metrics:
{'threshold': 0.5, 'accuracy': 0.7477876106194691, 'precision': 0.3333333333333333, 'recall': 0.008849557522123894, 'f1': 0.017241379310344827, 'roc_auc': 0.5908219720664792}

Ridge (L2) validation metrics:
{'threshold': 0.5, 'accuracy': 0.7472959685349065, 'precision': 0.3333333333333333, 'recall': 0.010816125860373648, 'f1': 0.02095238095238095, 'roc_auc': 0.5903917248145021}


In [37]:
"""2. Lasso coefficient importance"""
lasso_coefs = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lasso_model.coef_[0],
})

fig = plot_lasso_feature_importance(lasso_coefs, top_n=20)
fig.show()

embedded_lasso_features = lasso_coefs.loc[lasso_coefs["Coefficient"] != 0, "Feature"].tolist()
n_zeroed = int((lasso_model.coef_[0] == 0).sum())
print(f"Lasso zeroed-out features: {n_zeroed}")
print(f"Lasso-selected features (non-zero coefficient): {len(embedded_lasso_features)}")

Lasso zeroed-out features: 17
Lasso-selected features (non-zero coefficient): 58


In [38]:
"""3. Random Forest tree-based feature importance"""
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train_prep, y_train)

tree_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_,
}).sort_values("Importance", ascending=False)

mean_importance = float(rf_model.feature_importances_.mean())
fig = plot_tree_feature_importance(tree_importance_df, top_n=20, threshold=mean_importance)
fig.show()

embedded_tree_features = tree_importance_df.loc[
    tree_importance_df["Importance"] > mean_importance, "Feature"
].tolist()
print(f"Random Forest mean importance: {mean_importance:.4f}")
print(f"Tree-selected features (importance > mean): {len(embedded_tree_features)}")

Random Forest mean importance: 0.0133
Tree-selected features (importance > mean): 37


In [39]:
"""4. Embedded Methods — combined selection"""
embedded_selected_features = sorted(set(embedded_lasso_features) | set(embedded_tree_features))
print(f"Embedded-selected features (Lasso non-zero OR tree above mean): "
      f"{len(embedded_selected_features)} / {len(feature_names)}")

Embedded-selected features (Lasso non-zero OR tree above mean): 62 / 75


> **Main Insights — Embedded methods**  
Lasso eliminates a substantial share of the features by zeroing their coefficients, confirming that many one-hot dummies and redundant numeric ratios add little new signal once the strongest predictors are present. Ridge keeps every feature but shrinks them, and reaches comparable validation performance to Lasso — meaning the discarded features were not actively harmful, just redundant. The Random Forest importance ranking surfaces a partly different set of features, including some that interact non-linearly with the target and are therefore harder for the linear Lasso to pick up. Taking the union of Lasso and tree selections preserves both the linear and non-linear views.


### <h4 style="border-bottom: 2px solid #b79ad6; margin: 0 0 15px 0; padding-bottom:4px; font-weight:bold; font-size:1.3em;text-align:left;">9.4 Consensus Feature Selection</h4>

Each of the three method families produced an independent recommendation. Combining them by majority vote yields a **consensus set**: features supported by at least two of the three families (filter, wrapper, embedded). This balances complementary perspectives — statistical filters, model-driven RFE, and shrinkage/tree importances — and reduces the risk that a single method's blind spot drives the final selection.

The full breakdown plus the consensus list are saved to `project_data/selected_features.json` so downstream notebooks (`04_model_selection`, `05_final_submission`) can load the same selection without recomputing it.


In [40]:
"""1. Consensus across method families and persistence to JSON"""
method_sets = {
    "filter":   set(filter_selected_features),
    "wrapper":  set(wrapper_selected_features),
    "embedded": set(embedded_selected_features),
}

vote_counts = Counter()
for feature_set in method_sets.values():
    vote_counts.update(feature_set)

consensus_features = sorted([f for f, c in vote_counts.items() if c >= 2])

print("Selection summary (number of features per method family):")
for name, feature_set in method_sets.items():
    print(f"  {name:<9} {len(feature_set)}")
print(f"  consensus (>= 2 method families): {len(consensus_features)}")

selection_output = {
    "filter":         sorted(method_sets["filter"]),
    "wrapper":        sorted(method_sets["wrapper"]),
    "embedded":       sorted(method_sets["embedded"]),
    "embedded_lasso": sorted(embedded_lasso_features),
    "embedded_tree":  sorted(embedded_tree_features),
    "consensus":      consensus_features,
}

output_path = PROJECT_ROOT / "project_data" / "selected_features.json"
with open(output_path, "w") as fh:
    json.dump(selection_output, fh, indent=2)
print(f"\nSaved selected features to: {output_path.relative_to(PROJECT_ROOT)}")

Selection summary (number of features per method family):
  filter    41
  wrapper   60
  embedded  62
  consensus (>= 2 method families): 55

Saved selected features to: project_data\selected_features.json


The table below makes the per-method decision explicit, one row per feature. The first six columns show what each individual signal recommends (`Keep` or `Discard`); `Keep Votes` counts how many of those six signals were `Keep`; `Final Decision` is the family-level consensus from the previous cell — the one persisted to `selected_features.json`.

Recall the difference between `Keep Votes` and `Final Decision`:

- **`Keep Votes`** is a flat 0–6 count across the individual signals. It is informative for diagnosis (e.g. spotting features that only Lasso likes, or that only filter methods flagged), but it is not used to make the final call.
- **`Final Decision`** comes from the two-stage rule: the three families (filter / wrapper / embedded) each cast one `Keep` vote, and a feature is kept iff at least two of those three families agreed. This avoids over-weighting filter signals (which would otherwise dominate with three columns) and matches what downstream notebooks read from `selected_features.json`.

Rows are sorted with consensus-`Keep` features at the top, ties broken by `Keep Votes` so unanimous keeps surface first.


In [41]:
"""2. Method comparison table — Keep/Discard per method, Final Decision = consensus"""
def _to_decision(features_set):
    """Return ['Keep'/'Discard', ...] aligned with feature_names."""
    return ["Keep" if f in features_set else "Discard" for f in feature_names]

variance_kept = set(feature_names) - set(near_zero_var_features)

method_columns = {
    "Variance":      variance_kept,
    "Spearman":      set(spearman_selected),
    "ANOVA":         set(anova_selected),
    "RFE":           set(wrapper_selected_features),
    "Lasso":         set(embedded_lasso_features),
    "Random Forest": set(embedded_tree_features),
}

comparison_df = pd.DataFrame(
    {col: _to_decision(features_set) for col, features_set in method_columns.items()},
    index=feature_names,
)
comparison_df.index.name = "Feature"

comparison_df["Keep Votes"] = (comparison_df == "Keep").sum(axis=1)


comparison_df["Final Decision"] = [
    "Keep" if f in consensus_features else "Discard" for f in feature_names
]

comparison_df = comparison_df.sort_values(
    by=["Final Decision", "Keep Votes"],
    ascending=[False, False],  
    kind="stable",
)

n_keep = int((comparison_df["Final Decision"] == "Keep").sum())
n_discard = int((comparison_df["Final Decision"] == "Discard").sum())
print(f"Final Decision: Keep={n_keep}, Discard={n_discard}, total={len(comparison_df)}")

with pd.option_context("display.max_rows", None):
    display(comparison_df)

Final Decision: Keep=55, Discard=20, total=75


,Variance,Spearman,ANOVA,RFE,Lasso,Random Forest,Keep Votes,Final Decision
Feature,,,,,,,,
FILE_CARD_GIFT,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
LAST_GIFT_AMT,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
LIFETIME_CARD_PROM,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
LIFETIME_GIFT_COUNT,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
LIFETIME_MIN_GIFT_AMT,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
LIFETIME_PROM,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
MONTHS_SINCE_FIRST_GIFT,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
MONTHS_SINCE_LAST_GIFT,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep
RECENT_AVG_GIFT_AMT,Keep,Keep,Keep,Keep,Keep,Keep,6,Keep


> **Main Insights — Section 8 wrap-up**  
The three method families agree more than they disagree: a small core of features (donor giving history and recent-response indicators) is consistently selected by filter, wrapper, and embedded methods. The disagreements are concentrated on weaker signals — categorical dummies and engineered ratios — where tree-based and linear views naturally diverge. The consensus list captures the robust core and is the recommended starting point for downstream modeling. PCA remains available as an alternative compressed representation if model families that benefit from dense, low-dimensional input (e.g. KNN, MLP) are explored later.


## <h3 style="border-bottom: 4px solid #b79ad6; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">10. Exporting the final train and test datasets</h3>


In [42]:
X_train_clean.to_csv(PROJECT_ROOT / "project_data" / "X_train_clean.csv")
X_val_clean.to_csv(PROJECT_ROOT / "project_data" / "X_val_clean.csv")
X_test_clean.to_csv(PROJECT_ROOT / "project_data" / "X_test_clean.csv")

y_train.to_csv(PROJECT_ROOT / "project_data" / "y_train_clean.csv")
y_val.to_csv(PROJECT_ROOT / "project_data" / "y_val_clean.csv")